In [ ]:
catalog = dbutils.widgets.get("catalog")
gold_schema = dbutils.widgets.get("gold_schema")
warehouse_id = dbutils.widgets.get("warehouse_id")

dbutils.widgets.text("prefix", "")
prefix = dbutils.widgets.get("prefix")

In [0]:
# Pre-builds a fallback "Sunny Bay Sales Genie" agent so every participant has a
# working Genie Agent even if they do not finish building their own in Lab 4.
# Reproduces the finalized Lab 4 configuration: the metric view as data source,
# the business-context instructions (incl. the fiscal-year rule from Step 2) and
# the parameterized "online vs offline sales" trusted asset.
#
# Uses the Genie REST API via the SDK's low-level api_client so it works
# regardless of the databricks-sdk version available on the cluster.
#
# NOTE: Genie Spaces were renamed to Genie Agents, but the REST API still uses
# the historical /api/2.0/genie/spaces endpoints and genie/rooms/ URLs.
# Do not rename those paths or the serialized_space payload field.
import json
from uuid import uuid4

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

TITLE = f"Sunny Bay Sales Genie"
DESCRIPTION = (
    "Ask questions about Sunny Bay Roastery coffee sales, customers, products, "
    "and stores using governed metrics from the Sunny Bay metric view."
)
metric_view = f"{catalog}.{gold_schema}.{prefix}sm_fact_coffee_sales_genie"

# Business context + metric meanings + the fiscal-year rule added in Lab 4 Step 2.
instructions = """* business context: Sunny Bay Roastery is a coffee roastery with an online shop and brick-and-mortar stores that sells coffee and related products. Online and offline sales data exists in daily granularity and covers sales made on the website (online) and on physical stores (offline). Business users will ask questions about sales over certain time periods and impact of marketing campaigns.

* metric meanings:
    - total_orders = count of orders
    - total_gross_revenue_usd = revenue before VAT and costs.
    - total_profit_usd = revenue minus cost of goods and VAT.

* Prefer last 30 days when no date is specified

* as the Covid-19 pandemic had significant impact on the business, users might refer to it. it happend between early 2020 and early 2023

* if you can't provide an answer, state that cleary to avoid inaccurate results.

* whenever users are asking about dates, use the fiscal year instead of the calendar year. It starts on June 1st and ends on May 31st."""

# Trusted asset: a verified, parameterized query for a frequently asked question.
online_offline_sql = f"""SELECT
  CASE
    WHEN MONTH(`date`) >= 6 THEN YEAR(`date`) + 1
    ELSE YEAR(`date`)
  END AS fiscal_year,
  store_online,
  MEASURE(total_gross_revenue_usd) AS total_gross_revenue_usd
FROM
  {metric_view}
WHERE
  `date` IS NOT NULL
  AND store_online = :onlinesales
GROUP BY ALL
ORDER BY
  fiscal_year,
  store_online"""

serialized_space = {
    "version": 2,
    "config": {
        "sample_questions": [
            {
                "id": uuid4().hex,
                "question": ["Show me the profit by month for the year 2023 as a bar chart"],
            },
            {
                "id": uuid4().hex,
                "question": ["Show online and offline sales per year."],
            },
        ]
    },
    "data_sources": {
        "tables": [
            {"identifier": metric_view},
        ]
    },
    "instructions": {
        "text_instructions": [
            {"id": uuid4().hex, "content": [instructions]},
        ],
        "example_question_sqls": [
            {
                "id": uuid4().hex,
                "question": ["Show online and offline sales per year."],
                "sql": [online_offline_sql],
                "parameters": [
                    {
                        "name": "onlinesales",
                        "type_hint": "STRING",
                        "default_value": {"values": ["false"]},
                    }
                ],
                "usage_guidance": [
                    "Use this verified query to compare online vs offline gross revenue by fiscal year."
                ],
            }
        ],
    },
}

serialized_json = json.dumps(serialized_space)


def find_space_by_title(title):
    page_token = None
    while True:
        query = {"page_size": 200}
        if page_token:
            query["page_token"] = page_token
        resp = w.api_client.do("GET", "/api/2.0/genie/spaces", query=query)
        for space in (resp.get("spaces") or []):
            if space.get("title") == title:
                return space
        page_token = resp.get("next_page_token")
        if not page_token:
            return None


existing = find_space_by_title(TITLE)
if existing:
    space_id = existing["space_id"]
    w.api_client.do(
        "PATCH",
        f"/api/2.0/genie/spaces/{space_id}",
        body={
            "serialized_space": serialized_json,
            "title": TITLE,
            "description": DESCRIPTION,
            "warehouse_id": warehouse_id,
        },
    )
    operation = "updated"
else:
    resp = w.api_client.do(
        "POST",
        "/api/2.0/genie/spaces",
        body={
            "warehouse_id": warehouse_id,
            "serialized_space": serialized_json,
            "title": TITLE,
            "description": DESCRIPTION,
        },
    )
    space_id = resp.get("space_id") or resp.get("id")
    operation = "created"

print(f"\u2705 Fallback Genie Agent {operation}: {TITLE}")
print(f"   space_id: {space_id}")
print(f"   open it at: {w.config.host}/genie/rooms/{space_id}")